In [1]:
import numpy as np
import matplotlib.pyplot as plt
import fractions
import scipy
from matplotlib.collections import LineCollection

import jump_process

ModuleNotFoundError: No module named 'jump_process'

In [ ]:
rng = np.random.default_rng(seed=27)

In [ ]:
def make_line_collection(times, positions, velocity, *args, **kwargs):
    segment = [(times[0], positions[0])]
    for k in range(len(times) - 1):
        t1, t2 = times[k:k + 2]
        x1, x2 = positions[k: k + 2]

        segment.append((t2, x1 + velocity * t2))
        segment.append((t2, x2 + velocity * t2))

    return LineCollection([segment], *args, **kwargs)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(12, 4))

param_sets = [
    {"time_scale": 1.0, "size_scale": 1.0},
    {"time_scale": 0.2, "size_scale": 0.2},
    {"time_scale": 5.0, "size_scale": 5.0},
]

axes[0].set_ylabel("Terminus position")
axes[0].set_xlabel("Time")

for ax, params in zip(axes, param_sets):
    velocity = params["size_scale"] / params["time_scale"]
    final_time = 80

    generator = jump_process.CompoundGammaGenerator(rng=rng, **params)
    
    colors = ["tab:red", "tab:blue", "tab:orange", "tab:green"]
    for color in colors:
        ts, xs = jump_process.generate_path(generator=generator, final_time=final_time)
        collection = make_line_collection(ts, xs, velocity, color=color)
        ax.add_collection(collection)
    ax.autoscale_view()
    ax.set_xlim(-1, final_time)
    ax.set_xticks([])
    ax.set_yticks([])
    λ = fractions.Fraction(params["size_scale"]).limit_denominator(10)
    τ = fractions.Fraction(params["time_scale"]).limit_denominator(10)
    ax.set_title(f"event size = {λ}, wait time = {τ}", fontsize=11)

fig.savefig("compound-poisson.pdf", bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(12, 4))

param_sets = [
    {"time_scale": 1.0, "time_shape": 1.0, "size_scale": 1.0, "size_shape": 1.0},
    {"time_scale": 1.0 / 5, "time_shape": 5.0, "size_scale": 1.0, "size_shape": 1.0},
    {"time_scale": 5.0, "time_shape": 1.0 / 5, "size_scale": 1.0, "size_shape": 1.0},
]

axes[0].set_ylabel("Terminus position")
axes[0].set_xlabel("Time")

for ax, params in zip(axes, param_sets):
    velocity = params["size_scale"] * params["size_shape"] / (params["time_scale"] * params["time_shape"])
    final_time = 80 * params["time_scale"] * params["time_shape"]

    generator = jump_process.CompoundGammaGenerator(rng=rng, **params)
    
    colors = ["tab:red", "tab:blue", "tab:orange", "tab:green"]
    for color in colors:
        ts, xs = jump_process.generate_path(generator=generator, final_time=final_time)
        collection = make_line_collection(ts, xs, velocity, color=color)
        ax.add_collection(collection)
    ax.autoscale_view()
    ax.set_xlim(-1, final_time)
    ax.set_xticks([])
    ax.set_yticks([])
    α = fractions.Fraction(params["time_shape"]).limit_denominator(10)
    ax.set_title(f"shape = {α}", fontsize=11)

fig.savefig("compound-gamma.pdf", bbox_inches="tight")

In [ ]:
big_slow_params = {"time_scale": 10.0, "size_scale": 5.0}
big_slow = jump_process.CompoundGammaGenerator(rng=rng, **big_slow_params)
small_fast_params = {"time_scale": 0.5, "size_scale": 0.25}
small_fast = jump_process.CompoundGammaGenerator(rng=rng, **small_fast_params)

big_slow_velocity = big_slow_params["size_scale"] / big_slow_params["time_scale"]
small_fast_velocity = small_fast_params["size_scale"] / small_fast_params["time_scale"]
velocity = big_slow_velocity + small_fast_velocity

generator = jump_process.SumGenerator(rng=rng, generators=[big_slow, small_fast])

In [ ]:
fig, ax = plt.subplots() 

colors = ["tab:red", "tab:blue", "tab:orange", "tab:green"]
for color in colors:
    ts, xs = jump_process.generate_path(generator=generator, final_time=final_time)
    collection = make_line_collection(ts, xs, velocity, color=color)
    ax.add_collection(collection)

ax.set_xlabel("Time")
ax.set_ylabel("Terminus position")

ax.set_xticks([])
ax.set_yticks([])
ax.autoscale_view()
ax.set_xlim((-1, final_time))

fig.savefig("sum-compound.pdf", bbox_inches="tight")